In [ ]:
import os
from google.cloud import bigquery
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
import hail as hl
from pyspark.sql.functions import col
import pandas as pd
import json

my_bucket = os.getenv('WORKSPACE_BUCKET')

# Phenome Association Study

In [ ]:
def phecode_icd_query_with_string_matching():
    """Original approach - includes string matching fallback"""
    workspace_cdr = os.environ["WORKSPACE_CDR"]
    
    query = f"""
        SELECT DISTINCT
            person_id,
            date,
            concept_code AS ICD,
            vocabulary_id
        FROM (
            -- Concept ID joins
            SELECT
                co.person_id,
                co.condition_start_date AS date,
                c.concept_code,
                c.vocabulary_id
            FROM
                `{workspace_cdr}.condition_occurrence` co
            LEFT JOIN
                `{workspace_cdr}.concept` c
            ON
                co.condition_source_concept_id = c.concept_id
            WHERE
                c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
            
            UNION DISTINCT
            
            SELECT
                o.person_id,
                o.observation_date AS date,
                c.concept_code,
                c.vocabulary_id
            FROM
                `{workspace_cdr}.observation` o
            LEFT JOIN
                `{workspace_cdr}.concept` c
            ON
                o.observation_source_concept_id = c.concept_id
            WHERE
                c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
            
            UNION DISTINCT
            
            -- String matching fallback on source_value
            SELECT
                co.person_id,
                co.condition_start_date AS date,
                c.concept_code,
                c.vocabulary_id
            FROM
                `{workspace_cdr}.condition_occurrence` co
            INNER JOIN
                `{workspace_cdr}.concept` c
            ON
                co.condition_source_value = c.concept_code
            WHERE
                c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
            
            UNION DISTINCT
            
            SELECT
                o.person_id,
                o.observation_date AS date,
                c.concept_code,
                c.vocabulary_id
            FROM
                `{workspace_cdr}.observation` o
            INNER JOIN
                `{workspace_cdr}.concept` c
            ON
                o.observation_source_value = c.concept_code
            WHERE
                c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
        )
    """
    return query

my_bucket = os.getenv('WORKSPACE_BUCKET')

client = bigquery.Client()
query_job = client.query(phecode_icd_query_with_string_matching())
arrow_table = query_job.to_arrow()


pq.write_table(arrow_table, f'{my_bucket}/tmp/icds.parquet')

sc = hl.spark_context()
spark = SparkSession(sc)
spark_df = spark.read.parquet(f"{my_bucket}/tmp/icds.parquet")
spark_df = spark_df.withColumn("date", col("date").cast("string"))
ht = hl.Table.from_spark(spark_df)
ht = ht.repartition(10)
ht = ht.checkpoint(f'{my_bucket}/tmp/ICD_codes.ht', overwrite=True)

In [ ]:
# This query represents dataset "test5" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_81010780_survey_sql = """
    SELECT
    answer.person_id,
    CAST(DATE(MAX(answer.survey_datetime)) AS STRING) AS date 
FROM
    `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
WHERE
    (
        question_concept_id IN (SELECT
            DISTINCT concept_id                         
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c                         
        JOIN
            (SELECT
                CAST(cr.id as string) AS id                               
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr                               
            WHERE
                concept_id IN (1586134)                               
                AND domain_id = 'SURVEY') a 
                ON (c.path like CONCAT('%', a.id, '.%'))                         
        WHERE
            domain_id = 'SURVEY'                         
            AND type = 'PPI'                         
            AND subtype = 'QUESTION')
    )
GROUP BY answer.person_id"""

survery_time_df = pd.read_gbq(
    dataset_81010780_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

# This query represents dataset "test5" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_81010780_person_sql = """
    SELECT
        person.person_id,
        CAST(DATE(person.birth_datetime) AS STRING) AS date_of_birth,
        p_sex_at_birth_concept.concept_name AS sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id"""

person_df = pd.read_gbq(
    dataset_81010780_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
# This query represents dataset "EHRandWGS" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_34455153_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) )"""

include_ids = pd.read_gbq(
    dataset_34455153_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

has_WGS_EHR = set(include_ids['person_id'].to_list()) 

In [ ]:
ancestry_df = hl.import_table(
    'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv',
    delimiter='\t'
)
ancestry_df = ancestry_df.key_by(ancestry_df.research_id)

#========================================GET AGE PROXY=========================================
survery_time_df = hl.Table.from_pandas(survery_time_df)
survery_time = survery_time_df.annotate(
    date_long = hl.experimental.strptime(survery_time_df.date + " 00:00:00", "%Y-%m-%d %H:%M:%S", "America/New_York")
)
survery_time = survery_time.select('person_id', 'date_long')
survery_time = survery_time.checkpoint(f'{my_bucket}/tmp/survey.ht', overwrite=True)

max_date = hl.read_table(f'{my_bucket}/tmp/ICD_codes.ht')
max_date = max_date.annotate(date_long = hl.experimental.strptime(max_date.date + " 00:00:00", "%F %H:%M:%S", "America/New_York"))
max_date = max_date.select('person_id', 'date_long')
max_date = max_date.union(survery_time, unify=True)
max_date = max_date.group_by('person_id').aggregate(max_date = hl.agg.max(max_date.date_long))
max_date = max_date.checkpoint(f'{my_bucket}/tmp/max_date.ht', overwrite=True)
#========================================GET AGE PROXY=========================================

person_df = hl.Table.from_pandas(person_df)
person_df = person_df.annotate(
    dob_long = hl.experimental.strptime(person_df.date_of_birth + " 00:00:00", "%F %H:%M:%S", "America/New_York"),
    last_entry = max_date[hl.int64(person_df.person_id)].max_date
)
person_df = person_df.annotate(
    age_last_contact = hl.floor((person_df.last_entry - person_df.dob_long)/(60*60*24*365.25) + 0.5),
    sex_at_birth = hl.case() \
        .when(person_df.sex_at_birth == 'Female', 1)
        .when(person_df.sex_at_birth == 'Male', 0)
        .default(hl.missing(hl.tint)),
    ancestry_pcs = ancestry_df[hl.str(person_df.person_id)].pca_features.replace('\[','').replace('\]', '').split(',')
)

person_df = person_df.select('person_id', 'age_last_contact', 'sex_at_birth', 'ancestry_pcs')

person_df = person_df.annotate(
    **{f'pc{x + 1}':person_df.ancestry_pcs[x] for x in range(10)}
).drop('ancestry_pcs')

person_df = person_df.checkpoint(f'{my_bucket}/tmp/person_df_cpt.ht', overwrite=True)
# = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

sex = hl.import_table('gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv')
sex = sex.key_by('research_id').select('dragen_sex_ploidy')
sex_filter = sex.filter(((sex.dragen_sex_ploidy != 'XX') & (sex.dragen_sex_ploidy != 'XY')) | hl.is_missing(sex.dragen_sex_ploidy))
sex_filter = sex_filter.research_id.collect()

flagged_samples_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
flagged_samples = hl.import_table(flagged_samples_path)
flagged_sample_filter = flagged_samples.s.collect()

sample_filter = sex_filter + flagged_sample_filter

pop_ids = has_WGS_EHR - set(sample_filter)
pop_ids = hl.Table.parallelize(
    [{'person_id': int(x)} for x in pop_ids],
    schema=hl.tstruct(person_id=hl.tint32)
)
pop_ids = pop_ids.key_by('person_id')

# = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

person_df = person_df.filter(hl.is_defined(pop_ids[person_df.person_id]))
person_df = person_df.key_by('person_id')
pc_cols = [f'pc{i}' for i in range(1, 11)]
person_df = person_df.annotate(
    **{pc: hl.float64(person_df[pc]) for pc in pc_cols}
)
person_df = person_df.checkpoint(f'{my_bucket}/tmp/hmbs_demog.ht', overwrite=True)

In [ ]:
# # ====================================================================================
# !gsutil cp gs://zoghbi-lab-data/shahil/HMBS.ht.gz .
# !tar -xzvf HMBS.ht.gz
# !gsutil -m cp -r HMBS.ht {my_bucket}/tmp/HMBS.ht
# hmbs = hl.read_table(f'{my_bucket}/tmp/HMBS.ht')
# hmbs = hmbs.repartition(1)
# hmbs = hmbs.checkpoint(f'{my_bucket}/tmp/cptA.ht', overwrite=True)
# # ====================================================================================

# # ====================================================================================
# clinvar_file = 'clinvar_20251013.vcf.gz'
# # !wget https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/weekly/{clinvar_file}
# # !gsutil cp {clinvar_file} {my_bucket}/tmp/
# recode = {str(i): f"chr{i}" for i in list(range(1, 23)) + ['X', 'Y']}
# recode['MT'] = 'chrM'
# clinvar_data = hl.import_vcf(f'{my_bucket}/tmp/{clinvar_file}', reference_genome='GRCh38', contig_recoding=recode, skip_invalid_loci=True, force_bgz=True).rows()

# clinvar_data = clinvar_data.filter(
#     (clinvar_data.locus.contig == 'chr11') &
#     (clinvar_data.locus.position >= 119085034 - 50) & 
#     (clinvar_data.locus.position <= 119093285 + 50)
# )
# clinvar_data = clinvar_data.filter(
#     clinvar_data.info.CLNSIG.any(lambda x: x.contains("Pathogenic") | x.contains("Likely_pathogenic"))
# )

# clinvar_data = clinvar_data.checkpoint(f'{my_bucket}/tmp/clinvar.ht', overwrite=True)
# # ====================================================================================

# # ====================================================================================
# aou = hl.import_table('gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/vat/vat_complete.bgz.tsv.gz', force_bgz=True)
# cols = [col for col in aou.row if col.startswith('gvs_') or col == 'vid']
# aou = aou.select(*cols)
# aou = aou.key_by(aou.vid)
# aou = aou.distinct()
# aou = aou.checkpoint(f'{my_bucket}/data/vat.ht')
# aou = aou.annotate(locus = hl.locus('chr' + aou.vid.split('-')[0], hl.int(aou.vid.split('-')[1]), reference_genome='GRCh38'),
#                  alleles = hl.array(aou.vid.split('-')[2:])
#                 )
# aou = aou.key_by('locus', 'alleles')
# aou = aou.checkpoint(f'{my_bucket}/data/vat_keyed.ht', overwrite=True)
# # ====================================================================================
aou = hl.read_table(f'gs://fc-secure-57a37491-77aa-498c-b0f0-2163b5feb12e/data/vat_keyed.ht')
aou = aou.filter(
    (aou.locus.contig == 'chr11') &
    (aou.locus.position >= 119085034 - 50) & 
    (aou.locus.position <= 119093285 + 50)
)
aou = aou.checkpoint(f'{my_bucket}/tmp/cpt.ht', overwrite=True)
mt = hl.read_matrix_table('gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt')
hmbs = hl.read_table(f'{my_bucket}/tmp/cptA.ht')
clinvar_data = hl.read_table(f'{my_bucket}/tmp/clinvar.ht')

mt = mt.filter_rows(
    (mt.locus.contig == 'chr11') & 
        (
            (mt.locus.position >= (119085034 - 50)) & 
            (mt.locus.position <= (119093285 + 50))
        )
)

mt = mt.annotate_rows(
    region_type = hmbs[mt.row_key].region.split('-')[1],
    esm1b = hl.float64(hmbs[mt.row_key].esmlb),
    am_score = hl.float64(hmbs[mt.row_key].am_score),
    most_deleterious_consequence = hmbs[mt.row_key].most_deleterious_consequence_cds,
    splice_deltamax = hl.float(hmbs[mt.row_key].SpliceAI_DeltaMax),
    clinvar_path = hl.if_else(
        hl.is_defined(clinvar_data[mt.row_key]),
        1,
        0
    ),
    aou_ac = hl.int(aou[mt.row_key].gvs_all_ac)
)

mt = mt.repartition(1)

mt = mt.checkpoint(f'{my_bucket}/tmp/hmbs_annotated.mt', overwrite=True)
# = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 


# Exon boundaries (half-open intervals)
exons = [
    (119085034, 119085066), (119088255, 119088308), (119088635, 119088707),
    (119089082, 119089131), (119089217, 119089272), (119089683, 119089760),
    (119089990, 119090067), (119090190, 119090265), (119091413, 119091526),
    (119092125, 119092163), (119092404, 119092523), (119092758, 119092811),
    (119092935, 119093021), (119093110, 119093283)
]

# Exon intervals for frameshift checking
exon_intervals = hl.array([hl.interval(start, end) for start, end in exons])

# Splice site intervals (±2 from boundaries)
splice_intervals = []
for i, (start, end) in enumerate(exons):
    if i > 0:  # Acceptor site
        splice_intervals.append(hl.interval(start - 2, start + 2))
    if i < len(exons) - 1:  # Donor site
        splice_intervals.append(hl.interval(end - 2, end + 2))

splice_intervals = hl.array(splice_intervals)

# # Read and annotate
mt = mt.annotate_rows(
    ref_length = hl.len(mt.alleles[0]),
    alt_length = hl.len(mt.alleles[1]),
    var_end = mt.locus.position + hl.len(mt.alleles[0]),
)

# Combined filter: SNPs OR indels meeting criteria
mt = mt.filter_rows(
    # SNP criteria (any of 4 conditions)
    (
        (mt.ref_length == mt.alt_length) &  # SNPs only
        (
            ((mt.most_deleterious_consequence == 'missense_variant') & (mt.clinvar_path == 1)) |
            (mt.most_deleterious_consequence == 'stop_gained') |
            (mt.region_type == 'SpliceSite')
        )
    ) |
    # Indel criteria (splice disruption OR frameshift)
    (
        (mt.ref_length != mt.alt_length) &  # Indels only
        (
            # Splice disruption: variant span overlaps ±2 regions
            hl.any(splice_intervals.map(
                lambda si: (mt.locus.position < si.end) & (si.start < mt.var_end)
            )) |
            # Frameshift: in exon AND not multiple of 3
            (
                hl.any(exon_intervals.map(lambda x: x.contains(mt.locus.position))) &
                ((mt.ref_length - mt.alt_length) % 3 != 0)
            )
        )
    )
)

mt = mt.checkpoint(f'{my_bucket}/tmp/hmbs_filtered.mt', overwrite=True)


mt = mt.filter_rows((mt.aou_ac > 0) & (mt.aou_ac < 6))
mt = mt.annotate_rows(
    carriers = hl.agg.filter(
        mt.GT.is_non_ref(),
        hl.agg.collect(mt.s)
    )
)
pid_carriers = mt.aggregate_rows(hl.agg.collect(mt.carriers))
lst = []
for x in pid_carriers:
    for y in x:
        lst.append(y)
    
pid_carriers = hl.Table.parallelize(
    [{'person_id': int(x)} for x in lst],
    schema=hl.tstruct(person_id=hl.tint32)
)
pid_carriers = pid_carriers.key_by('person_id')

person_df = hl.read_table(f'{my_bucket}/tmp/hmbs_demog.ht')

pid_carriers = pid_carriers.filter(
    hl.is_defined(person_df[pid_carriers.person_id])
)

pid_carriers = pid_carriers.checkpoint(f'{my_bucket}/tmp/carriers.ht', overwrite=True)

In [ ]:
ht = hl.read_table(f'{my_bucket}/tmp/ICD_codes.ht')
person_df = hl.read_table(f'{my_bucket}/tmp/hmbs_demog.ht')
pid_carriers = hl.read_table(f'{my_bucket}/tmp/carriers.ht')

person_df = person_df.annotate(
    carrier = hl.if_else(hl.is_defined(pid_carriers[person_df.person_id]), 1, 0)
)
person_df.export(f'{my_bucket}/tmp/phewas_covariates.csv', delimiter = ',')
!gsutil cp {my_bucket}/tmp/phewas_covariates.csv .

ht = ht.filter(hl.is_defined(person_df[hl.int32(ht.person_id)]))

ht = ht.key_by("vocabulary_id", "ICD")
#========================================PHECODE MAPPING=========================================
filename = 'phecodeX_R_map.csv'
!wget https://raw.githubusercontent.com/PheWAS/PhecodeX/refs/heads/main/{filename}
!gsutil cp {filename} {my_bucket}/tmp

phecode_mapping = hl.import_table(f'{my_bucket}/tmp/{filename}', delimiter=',', quote='"')
phecode_mapping = phecode_mapping.select("code", "vocabulary_id", "phecode")
phecode_mapping = phecode_mapping.key_by("vocabulary_id", ICD = phecode_mapping.code)
phecode_mapping = phecode_mapping.checkpoint(f'{my_bucket}/tmp/phecode_mapping.ht', overwrite=True)
#========================================PHECODE MAPPING=========================================
ht = ht.join(phecode_mapping, how='inner')

phecode_counts = ht.group_by("person_id", "phecode").aggregate(
    cnt = hl.agg.count()
)

phecode_counts = phecode_counts.filter(phecode_counts.cnt > 1)

phecode_counts = phecode_counts.checkpoint(f'{my_bucket}/tmp/phecode_cases.ht', overwrite=True)

phecode_counts.export(f'{my_bucket}/tmp/phecode_cases.csv', delimiter=',')
!gsutil cp {my_bucket}/tmp/phecode_cases.csv .

phecode_counts = phecode_counts.annotate(
    carrier = hl.if_else(hl.is_defined(pid_carriers[hl.int32(phecode_counts.person_id)]), 1, 0),
    sex_at_birth = person_df[hl.int32(phecode_counts.person_id)].sex_at_birth
)

carrier_count = pid_carriers.count()
total_count = person_df.count()

phecode_counts = phecode_counts.group_by('phecode').aggregate(
    total_cases_males = hl.agg.count_where(phecode_counts.sex_at_birth == 0),
    total_cases_females = hl.agg.count_where(phecode_counts.sex_at_birth == 1),
    carrier_cases_males = hl.agg.count_where(
        (phecode_counts.carrier == 1) & (phecode_counts.sex_at_birth == 0)
    ),
    carrier_cases_females = hl.agg.count_where(
        (phecode_counts.carrier == 1) & (phecode_counts.sex_at_birth == 1)
    ),
    total_cases = hl.agg.count(),
    total_case_carriers = hl.agg.count_where(phecode_counts.carrier == 1),
    carrier_count = carrier_count,
    total_count = person_df.count()
    
)

filename = 'phecodeX_R_sex.csv'
!wget https://raw.githubusercontent.com/PheWAS/PhecodeX/refs/heads/main/{filename}
!gsutil cp {filename} {my_bucket}/tmp
phecode_sx_spec = hl.import_table(f'{my_bucket}/tmp/phecodeX_R_sex.csv', delimiter=',', quote='"')
phecode_sx_spec = phecode_sx_spec.key_by('phecode')
phecode_counts = phecode_counts.annotate(
    male_only = phecode_sx_spec[phecode_counts.phecode].male_only,
    female_only = phecode_sx_spec[phecode_counts.phecode].female_only
)

phecode_counts.export(f'{my_bucket}/tmp/phecode_meta.csv', delimiter = ',')
!gsutil cp {my_bucket}/tmp/phecode_meta.csv .

In [ ]:
code_block = """
#!/usr/bin/env Rscript
# -*- coding: utf-8 -*-

###############################################################################
# FIRTH'S LOGISTIC REGRESSION PHEWAS
#
# Expected input files (in working directory):
#
#   1) phewas_covariates.csv
#        - person_id
#        - carrier            (0/1)
#        - sex_at_birth       (0 = male, 1 = female)
#        - age_last_contact
#        - pc1 ... pc10
#
#   2) phecode_cases.csv
#        - person_id
#        - phecode
#      (one row per person–phecode case)
#
#   3) phecode_meta.csv
#        - phecode
#        - total_cases_males
#        - total_cases_females
#        - carrier_cases_males
#        - carrier_cases_females
#        - total_cases
#        - total_case_carriers
#        - carrier_count         (global)
#        - total_count           (global)
#        - male_only   ("TRUE"/"FALSE")
#        - female_only ("TRUE"/"FALSE")
#
# Thresholds (using phecode_meta):
#   - sex-specific (male_only OR female_only):
#         >= 3 carrier cases AND >= 65 total cases
#   - non–sex-specific:
#         >= 3 carrier cases AND >= 70 total cases
#
# Status:
#   - "success" : Firth converged
#   - "failed"  : Firth attempted but did not converge / no usable data
#   - "skipped" : Below thresholds; no Firth run, but counts filled from metadata
###############################################################################

library(logistf)
library(data.table)
library(future.apply)
library(progressr)

cat("================================================================================\\n")
cat("FIRTH'S LOGISTIC REGRESSION PHEWAS\\n")
cat("================================================================================\\n\\n")

# ============================================================================
# CONFIGURATION
# ============================================================================

N_CORES <- 18  # set number of parallel cores

options(progressr.enable = TRUE)
handlers(global = TRUE)
handlers("progress")

# Positive control phecode (AIP)
aip_code <- "GE_966.23"

# ============================================================================
# 1. LOAD DATA
# ============================================================================

cat("Loading data...\\n")

cov_df   <- fread("phewas_covariates.csv", data.table = FALSE)
cases_df <- fread("phecode_cases.csv",     data.table = FALSE)
phe_meta <- fread("phecode_meta.csv",      data.table = FALSE)

cat(sprintf("  Covariates: %d individuals, %d columns\\n", nrow(cov_df), ncol(cov_df)))
cat(sprintf("  Case rows:  %d\\n", nrow(cases_df)))
cat(sprintf("  Phecodes in meta table: %d\\n", nrow(phe_meta)))
cat(sprintf("  Carriers:   %d\\n\\n", sum(cov_df$carrier == 1, na.rm = TRUE)))

# ============================================================================
# 2. PRECOMPUTE SEX RESTRICTIONS & THRESHOLDS (VECTORIZED)
# ============================================================================

# Convert male_only / female_only to sex restriction code:
#   0 = male only
#   1 = female only
#   2 = both sexes / unrestricted
male_flag   <- phe_meta$male_only   == "TRUE"
female_flag <- phe_meta$female_only == "TRUE"

phe_meta$sex_restriction <- ifelse(
  male_flag,   0L,
  ifelse(female_flag, 1L, 2L)
)

# Only keep phecodes that actually appear in cases
phecodes_with_cases <- unique(cases_df$phecode)
phe_meta <- phe_meta[phe_meta$phecode %in% phecodes_with_cases, , drop = FALSE]

# Thresholds:
# Sex-specific (0 or 1):
#   >= 3 carrier cases AND >= 70 total cases
# Non–sex-specific (2):
#   >= 3 carrier cases AND >= 75 total cases
phe_meta$passes_threshold <- with(
  phe_meta,
  ifelse(
    sex_restriction == 0L,
      carrier_cases_males   >= 3 & total_cases_males   >= 70,
    ifelse(
      sex_restriction == 1L,
        carrier_cases_females >= 3 & total_cases_females >= 70,
        total_case_carriers   >= 3 & total_cases         >= 75
    )
  )
)

# Always force AIP to be tested (even if it fails thresholds)
phe_meta$passes_threshold[phe_meta$phecode == aip_code] <- TRUE

# Row names for fast lookup by phecode
rownames(phe_meta) <- phe_meta$phecode

# Phecode sets
eligible_phecodes <- phe_meta$phecode[phe_meta$passes_threshold]
skipped_phecodes  <- phe_meta$phecode[!phe_meta$passes_threshold]

cat(sprintf("  Eligible phecodes (above thresholds): %d\\n", length(eligible_phecodes)))
cat(sprintf("  Skipped phecodes (below thresholds):  %d\\n\\n", length(skipped_phecodes)))

# Build a "skipped" table for phecodes below thresholds
if (length(skipped_phecodes) > 0) {
  skipped_meta <- phe_meta[skipped_phecodes, , drop = FALSE]
  
  tc   <- skipped_meta$total_cases
  tcc  <- skipped_meta$total_case_carriers
  cc   <- skipped_meta$carrier_count
  totN <- skipped_meta$total_count
  
  n_cases            <- tc
  n_controls         <- totN - tc
  n_carrier_cases    <- tcc
  n_carrier_controls <- cc - tcc
  n_noncarrier_cases <- tc - tcc
  n_noncarrier_ctrls <- (totN - cc) - (tc - tcc)
  
  skipped_results <- data.frame(
    phecode            = skipped_meta$phecode,
    n_cases            = n_cases,
    n_controls         = n_controls,
    n_carrier_cases    = n_carrier_cases,
    n_carrier_controls = n_carrier_controls,
    n_noncarrier_cases = n_noncarrier_cases,
    n_noncarrier_controls = n_noncarrier_ctrls,
    sex_restriction    = skipped_meta$sex_restriction,
    method             = "firth",
    beta               = NA_real_,
    se                 = NA_real_,
    OR                 = NA_real_,
    OR_95CI_lower      = NA_real_,
    OR_95CI_upper      = NA_real_,
    p_value            = NA_real_,
    neg_log10_p        = NA_real_,
    converged          = NA,   # not run
    error              = "Below precomputed Hail thresholds; skipped",
    status             = "skipped",
    stringsAsFactors   = FALSE
  )
} else {
  skipped_results <- NULL
}

# ============================================================================
# 3. FUNCTIONS
# ============================================================================

run_fishers <- function(cov_df, cases_df, phecode, sex_restriction) {
  # Positive-control Fisher's exact test on AIP
  
  phe_cases <- cases_df[cases_df$phecode == phecode, , drop = FALSE]
  if (nrow(phe_cases) == 0) {
    return(NULL)
  }
  case_ids <- unique(phe_cases$person_id)
  
  # Apply sex restriction
  if (sex_restriction == 0L) {
    analysis_df <- cov_df[cov_df$sex_at_birth == 0, ]
  } else if (sex_restriction == 1L) {
    analysis_df <- cov_df[cov_df$sex_at_birth == 1, ]
  } else {
    analysis_df <- cov_df
  }
  
  if (nrow(analysis_df) == 0) {
    return(NULL)
  }
  
  y       <- as.integer(analysis_df$person_id %in% case_ids)
  carrier <- analysis_df$carrier
  
  valid_idx <- !is.na(y) & !is.na(carrier)
  y       <- y[valid_idx]
  carrier <- carrier[valid_idx]
  
  if (length(y) == 0) {
    return(NULL)
  }
  
  n_carrier_cases    <- sum(carrier == 1 & y == 1, na.rm = TRUE)
  n_carrier_controls <- sum(carrier == 1 & y == 0, na.rm = TRUE)
  n_noncarrier_cases <- sum(carrier == 0 & y == 1, na.rm = TRUE)
  n_noncarrier_ctrls <- sum(carrier == 0 & y == 0, na.rm = TRUE)
  
  contingency_table <- matrix(
    c(n_carrier_cases,    n_carrier_controls,
      n_noncarrier_cases, n_noncarrier_ctrls),
    nrow = 2,
    dimnames = list(
      carrier = c("1", "0"),
      outcome = c("case", "control")
    )
  )
  
  tryCatch({
    fisher_result <- fisher.test(contingency_table)
    
    or_val   <- fisher_result$estimate
    ci_lower <- fisher_result$conf.int[1]
    ci_upper <- fisher_result$conf.int[2]
    p_value  <- fisher_result$p.value
    
    data.frame(
      phecode            = phecode,
      n_cases            = n_carrier_cases + n_noncarrier_cases,
      n_controls         = n_carrier_controls + n_noncarrier_ctrls,
      n_carrier_cases    = n_carrier_cases,
      n_carrier_controls = n_carrier_controls,
      n_noncarrier_cases = n_noncarrier_cases,
      n_noncarrier_controls = n_noncarrier_ctrls,
      sex_restriction    = sex_restriction,
      method             = "fisher",
      beta               = NA_real_,
      se                 = NA_real_,
      OR                 = or_val,
      OR_95CI_lower      = ci_lower,
      OR_95CI_upper      = ci_upper,
      p_value            = p_value,
      neg_log10_p        = -log10(p_value),
      converged          = TRUE,
      error              = NA_character_,
      status             = "success",
      stringsAsFactors   = FALSE
    )
  }, error = function(e) {
    data.frame(
      phecode            = phecode,
      n_cases            = n_carrier_cases + n_noncarrier_cases,
      n_controls         = n_carrier_controls + n_noncarrier_ctrls,
      n_carrier_cases    = n_carrier_cases,
      n_carrier_controls = n_carrier_controls,
      n_noncarrier_cases = n_noncarrier_cases,
      n_noncarrier_controls = n_noncarrier_ctrls,
      sex_restriction    = sex_restriction,
      method             = "fisher",
      beta               = NA_real_,
      se                 = NA_real_,
      OR                 = NA_real_,
      OR_95CI_lower      = NA_real_,
      OR_95CI_upper      = NA_real_,
      p_value            = NA_real_,
      neg_log10_p        = NA_real_,
      converged          = FALSE,
      error              = as.character(e$message),
      status             = "failed",
      stringsAsFactors   = FALSE
    )
  })
}

run_firths <- function(cov_df, cases_df, phe_meta, phecode, sex_restriction) {
  # Firth's logistic regression for a single phecode.
  # phecode is already known to pass meta thresholds.
  
  if (!phecode %in% rownames(phe_meta)) {
    return(NULL)
  }
  
  phe_cases <- cases_df[cases_df$phecode == phecode, , drop = FALSE]
  if (nrow(phe_cases) == 0) {
    return(NULL)
  }
  case_ids <- unique(phe_cases$person_id)
  
  if (sex_restriction == 0L) {
    analysis_df <- cov_df[cov_df$sex_at_birth == 0, ]
    include_sex <- FALSE
  } else if (sex_restriction == 1L) {
    analysis_df <- cov_df[cov_df$sex_at_birth == 1, ]
    include_sex <- FALSE
  } else {
    analysis_df <- cov_df
    include_sex <- TRUE
  }
  
  if (nrow(analysis_df) == 0) {
    return(NULL)
  }
  
  y <- as.integer(analysis_df$person_id %in% case_ids)
  
  n_cases    <- sum(y == 1, na.rm = TRUE)
  n_controls <- sum(y == 0, na.rm = TRUE)
  
  carrier            <- analysis_df$carrier
  n_carrier_cases    <- sum(carrier == 1 & y == 1, na.rm = TRUE)
  n_carrier_controls <- sum(carrier == 1 & y == 0, na.rm = TRUE)
  n_noncarrier_cases <- sum(carrier == 0 & y == 1, na.rm = TRUE)
  n_noncarrier_ctrls <- sum(carrier == 0 & y == 0, na.rm = TRUE)
  
  too_few_case_ctrl <- (
    is.na(n_cases)    || is.na(n_controls) ||
    n_cases < 3       || n_controls < 3
  )
  
  if (too_few_case_ctrl) {
    return(data.frame(
      phecode            = phecode,
      n_cases            = n_cases,
      n_controls         = n_controls,
      n_carrier_cases    = n_carrier_cases,
      n_carrier_controls = n_carrier_controls,
      n_noncarrier_cases = n_noncarrier_cases,
      n_noncarrier_controls = n_noncarrier_ctrls,
      sex_restriction    = sex_restriction,
      method             = "firth",
      beta               = NA_real_,
      se                 = NA_real_,
      OR                 = NA_real_,
      OR_95CI_lower      = NA_real_,
      OR_95CI_upper      = NA_real_,
      p_value            = NA_real_,
      neg_log10_p        = NA_real_,
      converged          = FALSE,
      error              = "Too few cases/controls in analysis subset; test not run",
      status             = "failed",
      stringsAsFactors   = FALSE
    ))
  }
  
  pc_vars <- paste0("pc", 1:10)

  # We will include age_last_contact and age_last_contact^2 (age_sq)
  if (include_sex) {
    covariates <- c("age_last_contact", "age_sq", pc_vars, "sex_at_birth")
  } else {
    covariates <- c("age_last_contact", "age_sq", pc_vars)
  }

  # Add age_sq to the analysis_df-derived data
  age_sq <- analysis_df$age_last_contact^2

  model_data <- data.frame(
    y         = y,
    carrier   = carrier,
    age_last_contact = analysis_df$age_last_contact,
    age_sq    = age_sq,
    analysis_df[, pc_vars, drop = FALSE]
  )

  if (include_sex) {
    model_data$sex_at_birth <- analysis_df$sex_at_birth
  }

  formula_str <- paste("y ~ carrier +", paste(covariates, collapse = " + "))
  formula_obj <- as.formula(formula_str)

  model_data <- na.omit(model_data)

  
  if (nrow(model_data) == 0) {
    return(data.frame(
      phecode            = phecode,
      n_cases            = n_cases,
      n_controls         = n_controls,
      n_carrier_cases    = n_carrier_cases,
      n_carrier_controls = n_carrier_controls,
      n_noncarrier_cases = n_noncarrier_cases,
      n_noncarrier_controls = n_noncarrier_ctrls,
      sex_restriction    = sex_restriction,
      method             = "firth",
      beta               = NA_real_,
      se                 = NA_real_,
      OR                 = NA_real_,
      OR_95CI_lower      = NA_real_,
      OR_95CI_upper      = NA_real_,
      p_value            = NA_real_,
      neg_log10_p        = NA_real_,
      converged          = FALSE,
      error              = "All rows removed by na.omit; test not run",
      status             = "failed",
      stringsAsFactors   = FALSE
    ))
  }
  
  result <- tryCatch({
    fit <- logistf(formula_obj, data = model_data, pl = TRUE)
    
    coef_idx <- 2  # "carrier"
    
    beta     <- fit$coefficients[coef_idx]
    se       <- sqrt(diag(fit$var)[coef_idx])
    ci_lower <- fit$ci.lower[coef_idx]
    ci_upper <- fit$ci.upper[coef_idx]
    p_value  <- fit$prob[coef_idx]
    
    or_val      <- exp(beta)
    or_ci_lower <- exp(ci_lower)
    or_ci_upper <- exp(ci_upper)
    neg_log10_p <- -log10(p_value)
    
    data.frame(
      phecode            = phecode,
      n_cases            = n_cases,
      n_controls         = n_controls,
      n_carrier_cases    = n_carrier_cases,
      n_carrier_controls = n_carrier_controls,
      n_noncarrier_cases = n_noncarrier_cases,
      n_noncarrier_controls = n_noncarrier_ctrls,
      sex_restriction    = sex_restriction,
      method             = "firth",
      beta               = beta,
      se                 = se,
      OR                 = or_val,
      OR_95CI_lower      = or_ci_lower,
      OR_95CI_upper      = or_ci_upper,
      p_value            = p_value,
      neg_log10_p        = neg_log10_p,
      converged          = TRUE,
      error              = NA_character_,
      status             = "success",
      stringsAsFactors   = FALSE
    )
  }, error = function(e) {
    data.frame(
      phecode            = phecode,
      n_cases            = n_cases,
      n_controls         = n_controls,
      n_carrier_cases    = n_carrier_cases,
      n_carrier_controls = n_carrier_controls,
      n_noncarrier_cases = n_noncarrier_cases,
      n_noncarrier_controls = n_noncarrier_ctrls,
      sex_restriction    = sex_restriction,
      method             = "firth",
      beta               = NA_real_,
      se                 = NA_real_,
      OR                 = NA_real_,
      OR_95CI_lower      = NA_real_,
      OR_95CI_upper      = NA_real_,
      p_value            = NA_real_,
      neg_log10_p        = NA_real_,
      converged          = FALSE,
      error              = as.character(e$message),
      status             = "failed",
      stringsAsFactors   = FALSE
    )
  })
  
  result
}

# ============================================================================
# 4. RUN PHEWAS
# ============================================================================

cat("================================================================================\\n")
cat("RUNNING PHEWAS\\n")
cat("================================================================================\\n\\n")

# --- Positive control: AIP with Fisher's exact test -------------------------

aip_result <- NULL
if (aip_code %in% phe_meta$phecode) {
  cat(sprintf("Testing positive control: %s (AIP) with Fisher's exact test...\\n", aip_code))
  
  sex_restriction_aip <- phe_meta[aip_code, "sex_restriction"]
  
  aip_result <- run_fishers(cov_df, cases_df, aip_code, sex_restriction_aip)
  
  if (!is.null(aip_result) && aip_result$converged) {
    cat(sprintf("  ✓ OR = %.1f, p = %.2e\\n", aip_result$OR, aip_result$p_value))
    cat(sprintf("  Cases: %d carriers / %d non-carriers\\n\\n",
                aip_result$n_carrier_cases, aip_result$n_noncarrier_cases))
  } else if (!is.null(aip_result)) {
    cat(sprintf("  ✗ Error: %s\\n\\n", aip_result$error))
  } else {
    cat("  ✗ AIP result is NULL (no data?)\\n\\n")
  }
} else {
  cat(sprintf("AIP code %s not found in meta; skipping positive control.\\n\\n", aip_code))
}

# --- Discovery phecodes (Firth) ---------------------------------------------

discovery_phecodes <- setdiff(eligible_phecodes, aip_code)

cat(sprintf("Testing %d discovery phecodes with %d cores...\\n\\n",
            length(discovery_phecodes), N_CORES))

plan(multisession, workers = N_CORES)

results_list <- with_progress({
  p <- progressor(along = discovery_phecodes)
  
  future_lapply(
    discovery_phecodes,
    function(phecode, cov_df, cases_df, phe_meta, p) {
      sex_restriction <- phe_meta[phecode, "sex_restriction"]
      res <- run_firths(cov_df, cases_df, phe_meta, phecode, sex_restriction)
      p(message = phecode)
      res
    },
    cov_df   = cov_df,
    cases_df = cases_df,
    phe_meta = phe_meta,
    p        = p,
    future.seed = TRUE
  )
})

plan(sequential)

results_list <- results_list[!sapply(results_list, is.null)]

results_df <- if (length(results_list) > 0) {
  do.call(rbind, results_list)
} else {
  data.frame()
}

successful <- if (nrow(results_df) > 0) {
  results_df[results_df$status == "success", , drop = FALSE]
} else {
  results_df[FALSE, , drop = FALSE]
}

failed <- if (nrow(results_df) > 0) {
  results_df[results_df$status == "failed", , drop = FALSE]
} else {
  results_df[FALSE, , drop = FALSE]
}

cat(sprintf("\\nFirth fits: %d success, %d failed\\n",
            nrow(successful), nrow(failed)))
cat(sprintf("Skipped (threshold): %d\\n\\n",
            ifelse(is.null(skipped_results), 0L, nrow(skipped_results))))

# ============================================================================
# 5. APPLY FDR CORRECTION (ONLY TO SUCCESSFUL FIRTH FITS)
# ============================================================================

if (nrow(successful) > 0) {
  cat("================================================================================\\n")
  cat("FDR CORRECTION (BENJAMINI-HOCHBERG)\\n")
  cat("================================================================================\\n\\n")
  
  successful$q_value         <- p.adjust(successful$p_value, method = "BH")
  successful$fdr_significant <- successful$q_value < 0.05
  successful$fdr_suggestive  <- successful$q_value < 0.2
  
  successful <- successful[order(successful$p_value), ]
  
  n_sig_05 <- sum(successful$q_value < 0.05, na.rm = TRUE)
  n_sig_20 <- sum(successful$q_value < 0.20, na.rm = TRUE)
  
  cat(sprintf("Significant at FDR q < 0.05: %d\\n", n_sig_05))
  cat(sprintf("Significant at FDR q < 0.20: %d\\n\\n", n_sig_20))
}

# Make sure skipped_results exists with proper type even if NULL
if (is.null(skipped_results)) {
  skipped_results <- data.frame()
}

# ============================================================================
# 6. HARMONIZE COLUMNS ACROSS TABLES
# ============================================================================

# Canonical column order for all result-like tables
result_cols <- c(
  "phecode",
  "n_cases",
  "n_controls",
  "n_carrier_cases",
  "n_carrier_controls",
  "n_noncarrier_cases",
  "n_noncarrier_controls",
  "sex_restriction",
  "method",
  "beta",
  "se",
  "OR",
  "OR_95CI_lower",
  "OR_95CI_upper",
  "p_value",
  "neg_log10_p",
  "converged",
  "error",
  "status",
  "q_value",
  "fdr_significant",
  "fdr_suggestive",
  "positive_control"
)

add_missing_cols <- function(df, cols) {
  if (nrow(df) == 0) {
    out <- as.data.frame(matrix(nrow = 0, ncol = length(cols)))
    names(out) <- cols
    return(out)
  }
  for (nm in cols) {
    if (!nm %in% colnames(df)) {
      df[[nm]] <- NA
    }
  }
  df[, cols, drop = FALSE]
}

successful      <- add_missing_cols(successful,      result_cols)
failed          <- add_missing_cols(failed,          result_cols)
skipped_results <- add_missing_cols(skipped_results, result_cols)

if (nrow(successful) > 0) {
  successful$status <- "success"
}
if (nrow(failed) > 0) {
  failed$status <- "failed"
}
if (nrow(skipped_results) > 0) {
  skipped_results$status <- "skipped"
}

# ============================================================================
# 7. MERGE SUCCESSFUL + SKIPPED + AIP
# ============================================================================

combined <- rbind(successful, skipped_results)

# Handle AIP (positive control)
if (!is.null(aip_result) && nrow(aip_result) == 1) {
  aip_result <- add_missing_cols(aip_result, result_cols)
  aip_result$status <- ifelse(isTRUE(aip_result$converged), "success", "failed")
  
  aip_result$positive_control <- TRUE
  aip_result$q_value          <- NA_real_
  aip_result$fdr_significant  <- NA
  aip_result$fdr_suggestive   <- NA
  
  if (nrow(combined) > 0) {
    combined$positive_control[is.na(combined$positive_control)] <- FALSE
  }
  combined <- rbind(aip_result, combined)
} else if (nrow(combined) > 0) {
  combined$positive_control[is.na(combined$positive_control)] <- FALSE
}

# ============================================================================
# 8. SAVE RESULTS
# ============================================================================

write.csv(combined, "phewas_firths_results.csv", row.names = FALSE)
cat("✓ Results (success + skipped + AIP) saved to: phewas_firths_results.csv\\n")

if (nrow(failed) > 0) {
  write.csv(failed, "phewas_failed_tests.csv", row.names = FALSE)
  cat("✓ Failed tests (non-converged / insufficient data) saved to: phewas_failed_tests.csv\\n")
}

cat("\\n================================================================================\\n")
cat("PHEWAS COMPLETE\\n")
cat("================================================================================\\n")
"""

with open("RunFirths.R", "w") as f:
    f.write(code_block)


In [ ]:
!gsutil cp {my_bucket}/tmp/phewas_covariates.csv .
!gsutil cp {my_bucket}/tmp/phecode_cases.csv .
!gsutil cp {my_bucket}/tmp/phecode_meta.csv .
!Rscript RunFirths.R

!wget https://raw.githubusercontent.com/PheWAS/PhecodeX/refs/heads/main/phecodeX_R_labels.csv
results = pd.read_csv('phewas_firths_results.csv')
labels = pd.read_csv('phecodeX_R_labels.csv')
# Merge on the phecode/phenotype columns to align descriptions
results = results.merge(labels[['phenotype', 'description']],
                        left_on='phecode', right_on='phenotype', how='left')

# (Optional) remove the now-redundant 'phenotype' column
results.drop('phenotype', axis=1, inplace=True)
results.to_csv('SupplementaryTable1.csv', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load data
df = pd.read_csv('SupplementaryTable1.csv')

# Remove AIP
df = df[df['phecode'] != 'GE_966.23'].copy()

# Extract phecode class
df['phecode_class'] = df['phecode'].str.split('_').str[0]

# mapping EM/GU together, GI/ID together
map_classes = {
    'EM': 'EM_GU',
    'GU': 'EM_GU',
    'GI': 'GI_ID',
    'ID': 'GI_ID'
}

df['phecode_class'] = df['phecode_class'].replace(map_classes)

# Define phecode class order
class_order = ['CA', 'NS', 'CV', 'EM_GU', 'MS', 'GI_ID', 'MB']


# Create categorical type for sorting
df['phecode_class_cat'] = pd.Categorical(df['phecode_class'], categories=class_order, ordered=True)

# Define three panels
panel1_df = df[df['q_value'] <= 0.05].copy()
panel2_df = df[(df['q_value'] > 0.05) & (df['q_value'] <= 0.20)].copy()
selected_bottom = ['MB_288.3', 'MB_286.2', 'MB_280.9', 'GI_527']
panel3_df = df[df['phecode'].isin(selected_bottom)].copy()

# Sort each panel by phecode class
panel1_df = panel1_df.sort_values(['phecode_class_cat', 'OR'])
panel2_df = panel2_df.sort_values(['phecode_class_cat', 'OR'])
panel3_df = panel3_df.sort_values(['phecode_class_cat', 'OR'])

# Define colors for each phecode class
colors = {
    'CA': '#e41a1c',
    'NS': '#377eb8',
    'CV': '#4daf4a',
    'EM_GU': '#984ea3',
    'GI_ID': '#ff7f00',
    'MB': '#000000',
    'MS': '#999999'
}

# Create figure
fig = plt.figure(figsize=(12, 10))

# Calculate positions for each panel
n1 = len(panel1_df)
n2 = len(panel2_df)
n3 = len(panel3_df)

# === KEY CHANGE: Separate control for visual density ===
# row_height_factor controls how much vertical space each row gets on the figure
# Set to 0.8 for 80% of original spacing (i.e., 20% tighter)
row_height_factor = 0.5

# Divider space scales with row height
divider_space = 0.05 * row_height_factor

# Total height uses row_height_factor for axis sizing
total_height = n1 * row_height_factor + divider_space + n2 * row_height_factor + divider_space + n3 * row_height_factor

# Normalize positions (0 to 1)
panel1_height = (n1 * row_height_factor) / total_height
panel2_height = (n2 * row_height_factor) / total_height
panel3_height = (n3 * row_height_factor) / total_height
divider_height = divider_space / total_height

# Bottom positions
panel3_bottom = 0.02 + 0.06 * row_height_factor
divider2_bottom = panel3_bottom + panel3_height
panel2_bottom = divider2_bottom + divider_height
divider1_bottom = panel2_bottom + panel2_height
panel1_bottom = divider1_bottom + divider_height

# Create axes
ax1 = fig.add_axes([0.25, panel1_bottom, 0.7, panel1_height])
ax2 = fig.add_axes([0.25, panel2_bottom, 0.7, panel2_height])
ax3 = fig.add_axes([0.25, panel3_bottom, 0.7, panel3_height])

def plot_panel(ax, data, y_start=0):
    """Plot forest plot for a panel"""
    y_positions = []
    labels = []
    
    for idx, (_, row) in enumerate(data.iterrows()):
        # === KEY CHANGE: Use integer spacing (0, 1, 2, 3, ...) ===
        y_pos = y_start + idx
        y_positions.append(y_pos)
        labels.append(row['description'])
        
        color = colors[row['phecode_class']]
        
        # Plot OR point estimate
        ax.scatter(row['OR'], y_pos, color=color, s=100, zorder=3, edgecolors='black', linewidths=0.5)
        
        # Plot 95% CI
        ax.plot([row['OR_95CI_lower'], row['OR_95CI_upper']], [y_pos, y_pos], 
                color=color, linewidth=2, zorder=2)
    
    return y_positions, labels

# Plot each panel
y_pos1, labels1 = plot_panel(ax1, panel1_df, 0)
y_pos2, labels2 = plot_panel(ax2, panel2_df, 0)
y_pos3, labels3 = plot_panel(ax3, panel3_df, 0)

# Set x-axis limits
all_data = pd.concat([panel1_df, panel2_df, panel3_df])
max_ci = all_data['OR_95CI_upper'].max()
x_upper = max(max_ci * 1.01, 4.0)
xticks = np.arange(0, np.ceil(x_upper) + 1)
for ax in (ax1, ax2, ax3):
    ax.set_xlim(0, x_upper)
    ax.set_xticks(xticks)

# Configure axes
for ax, y_pos, labels in [(ax1, y_pos1, labels1), (ax2, y_pos2, labels2), (ax3, y_pos3, labels3)]:
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=13.5)
    # === KEY CHANGE: Y-limits based on integer count ===
    ax.set_ylim(-0.5, len(labels) - 0.5)
    # Longer dash at x=1
    ax.axvline(x=1, color='gray', linestyle=(0, (8, 2)), linewidth=1.2, alpha=0.6)
    # Custom grid: lighter lines every 1, slightly darker every 5
    grid_ticks = np.arange(0, np.ceil(x_upper) + 1)
    for gx in grid_ticks:
        if gx % 5 == 0:
            ax.axvline(x=gx, color='gray', linestyle='-', linewidth=0.8, alpha=0.3, zorder=0)
        else:
            ax.axvline(x=gx, color='gray', linestyle='-', linewidth=0.6, alpha=0.1, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Invert y-axis so first entry is at top
    ax.invert_yaxis()

# Remove x-axis from top and middle panels
ax1.set_xlabel('')
ax1.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax2.set_xlabel('')
ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

# Only show x-axis on bottom panel
ax3.set_xlabel('Odds Ratio (95% CI)', fontsize=16.5)

# Add black divider bars using figure coordinates
divider1_y = divider1_bottom + divider_height/2
divider2_y = divider2_bottom + divider_height/2

# Draw thick black horizontal lines as dividers
fig.add_artist(plt.Line2D([0.25, 0.95], [divider1_y, divider1_y], 
                          color='black', linewidth=2, transform=fig.transFigure))
fig.add_artist(plt.Line2D([0.25, 0.95], [divider2_y, divider2_y], 
                          color='black', linewidth=2, transform=fig.transFigure))

# Add panel labels near divider bars
label_x = 0.92
label_offset = 0.04
bbox_dict=dict(
    boxstyle='round,pad=0.3',   # rounded corners
    facecolor='whitesmoke',          # box fill color
    edgecolor='grey',          # box border color
    linewidth=1.0
)
fig.text(label_x, divider1_y + divider_height/2 + label_offset, 'FDR < 0.05',
         fontsize=15, weight='bold', va='bottom', ha='right', bbox=bbox_dict)
fig.text(label_x, divider1_y - divider_height/2 - label_offset, '0.05 ≤ FDR < 0.20',
         fontsize=15, weight='bold', va='top', ha='right', bbox=bbox_dict)
fig.text(label_x, divider2_y - divider_height/2 - label_offset, 'FDR ≥ 0.20 (4 of 98)',
         fontsize=15, weight='bold', va='top', ha='right', bbox=bbox_dict)

# Create legend
legend_items = [
    ('Cancer', colors['CA']),
    ('Neurological', colors['NS']),
    ('Cardiovascular', colors['CV']),
    ('Metabolic/Genitourinary', colors['EM_GU']),
    ('Gastrointestinal/Infection', colors['GI_ID']),
    ('Psychiatric', colors['MB']),
    ('Musculoskeletal', colors['MS'])
]
legend_elements = [mpatches.Patch(facecolor=color, edgecolor='black', label=label) 
                   for label, color in legend_items]
legend_y = panel2_bottom + panel2_height * 0.60
legend = fig.legend(handles=legend_elements, loc='center right', title='Phecode Class', 
                    frameon=True, fontsize=13, bbox_to_anchor=(0.92, legend_y), 
                    bbox_transform=fig.transFigure, title_fontsize=14, fancybox=True)
frame = legend.get_frame()
frame.set_edgecolor('grey')
# Darken and slightly thicken the legend border for better contrast
legend.get_frame().set_edgecolor('gray')
legend.get_frame().set_linewidth(1.5)

plt.savefig('figure2.svg', dpi=300, bbox_inches='tight')

# Distribution of signficant phecodes among carriers

In [ ]:
cov_df = hl.import_table(f'{my_bucket}/tmp/phewas_covariates.csv', delimiter=',')

pid_carriers = hl.set(cov_df.aggregate(hl.agg.filter(hl.int(cov_df.carrier) == 1, hl.agg.collect(hl.int(cov_df.person_id)))))

phecode_counts = hl.read_table(f'{my_bucket}/tmp/phecode_cases.ht')

phecode_counts = phecode_counts.annotate(
    carrier = hl.if_else(pid_carriers.contains(hl.int32(phecode_counts.person_id)), 1, 0),
)

results = pd.read_csv('phewas_firths_results.csv')
results = results[results['q_value'] < 0.2]
results = results[results['n_carrier_cases'] < 5]


suggestive_phecodes = hl.set(results['phecode'].to_list())

phecode_counts = phecode_counts.filter(
    (phecode_counts.carrier == 1) &
    (suggestive_phecodes.contains(phecode_counts.phecode))
)

contributing_carriers = phecode_counts.aggregate(hl.len(hl.array(hl.agg.collect_as_set(phecode_counts.person_id))))

print(f"{contributing_carriers/hl.eval(hl.len(pid_carriers))*100:.0f}% of carriers contributed to {hl.eval(hl.len(suggestive_phecodes))} significant and suggestive phenotypes driven by 3 or 4 individuals")


# Prevalence of Psych phecodes among cases and carriers

In [ ]:
from scipy.stats import fisher_exact
import numpy as np

psych_phecodes = hl.set({
    "MB_280", "MB_280.2", "MB_280.21", "MB_280.3", "MB_280.31", "MB_280.5", "MB_280.51",
    "MB_280.8", "MB_280.9", "MB_280.91", "MB_283", "MB_284.1", "MB_284.2", "MB_286",
    "MB_286.1", "MB_286.11", "MB_286.2", "MB_286.3", "MB_288", "MB_288.3", "MB_289",
    "MB_290.1", "MB_296", "MB_299", "MB_308.1", "NS_329.8", "NS_333.2", "SS_807"
})

cov_df = hl.import_table(f'{my_bucket}/tmp/phewas_covariates.csv', delimiter=',')

pid_carriers = hl.set(cov_df.aggregate(hl.agg.filter(hl.int(cov_df.carrier) == 1, hl.agg.collect(hl.int(cov_df.person_id)))))

phecode_counts = hl.read_table(f'{my_bucket}/tmp/phecode_cases.ht')

phecode_counts = phecode_counts.annotate(
    carrier = hl.if_else(pid_carriers.contains(hl.int32(phecode_counts.person_id)), 1, 0),
)

phecode_counts = phecode_counts.filter(
    psych_phecodes.contains(phecode_counts.phecode)
)

carrier_with_pheno = len(list(phecode_counts.aggregate(hl.agg.filter(phecode_counts.carrier == 1, hl.agg.collect_as_set(phecode_counts.person_id)))))
carrier_without_pheno = hl.eval(hl.len((pid_carriers))) - carrier_with_pheno
case_prevalence = carrier_with_pheno/hl.eval(hl.len((pid_carriers)))
noncarrier_with_pheno = len(list(phecode_counts.aggregate(hl.agg.filter(phecode_counts.carrier == 0, hl.agg.collect_as_set(phecode_counts.person_id)))))
noncarrier_without_pheno = (cov_df.count() - hl.eval(hl.len((pid_carriers)))) - noncarrier_with_pheno  # 187394
control_prevalence = noncarrier_with_pheno/(cov_df.count() - hl.eval(hl.len((pid_carriers))))

# Build 2x2 contingency table
# Format: [[carrier_with, carrier_without],
#          [noncarrier_with, noncarrier_without]]
table = np.array([
    [carrier_with_pheno, carrier_without_pheno],
    [noncarrier_with_pheno, noncarrier_without_pheno]
])

# Fisher's exact test
odds_ratio, p_value = fisher_exact(table, alternative='two-sided')

# 95% CI for OR (using log transformation method)
a, b, c, d = carrier_with_pheno, carrier_without_pheno, noncarrier_with_pheno, noncarrier_without_pheno
log_or = np.log(odds_ratio)
se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_lower = np.exp(log_or - 1.96 * se_log_or)
ci_upper = np.exp(log_or + 1.96 * se_log_or)

# Results
print(f"Fisher's Exact Test p-value: {p_value:.4e}")
print(f"Odds Ratio: {odds_ratio:.3f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")
print(f"Case prevalence: {case_prevalence*100:.2f}%")
print(f"Control prevalence: {control_prevalence*100:.2f}%")


# Figure 1 Example

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Data
carrier_cases = 4
carrier_total = 41
noncarrier_cases = 20025
noncarrier_total = 317928

carrier_pct = (carrier_cases / carrier_total) * 100
noncarrier_pct = (noncarrier_cases / noncarrier_total) * 100

# Create figure
fig, ax = plt.subplots(figsize=(5.5, 5))

# Bar positions and values
categories = ['Carriers', 'Non-Carriers']
percentages = [carrier_pct, noncarrier_pct]
colors = ['#A81818', '#383838']

# Create bars
bars = ax.bar(categories, percentages, color=colors, width=0.8)

# Add percentage labels on bars

# Add percentage labels on bars
ax.text(0, carrier_pct + 0.3, f'{carrier_pct:.1f}%\n({carrier_cases}/{carrier_total})', 
        ha='center', va='bottom', fontsize=23)
ax.text(1, noncarrier_pct + 0.3, f'{noncarrier_pct:.1f}%\n({noncarrier_cases*1e-3:.1f}k/{noncarrier_total*1e-3:.1f}k)', 
        ha='center', va='bottom', fontsize=23)

# Styling
ax.set_title('% Diagnosed with Secondary\nMalignant Neoplasm', fontsize=27, pad=50)
ax.tick_params(axis='x', labelsize=27) # Set labelsize to your desired font size
ax.yaxis.set_visible(False)

ax.set_ylim(0, 13)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)


plt.tight_layout()
plt.savefig('my_plot.svg', format='svg', transparent=True) 
plt.show()
plt.close()

# Biochemical testing among individuals with AIP diagnosis

In [ ]:
import pandas
import os

# This query represents dataset "PBG_ALA" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_43814451_measurement_sql = """
    SELECT
        measurement.person_id,
        m_standard_concept.concept_name as standard_concept_name,
        measurement.measurement_datetime,
        measurement.value_as_number,
        m_unit.concept_name as unit_concept_name,
        m_source_concept.concept_name as source_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (40789134)      
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN (4120604) 
                            AND is_standard = 1 )) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (40789134)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
            ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

dataset_43814451_measurement_df = pandas.read_gbq(
    dataset_43814451_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")



In [ ]:
# ============================================================================
# SUPPLEMENTARY TABLE 2: Individual-level summary of AIP cases, carriers, and biochemical testing
# ============================================================================

import pandas as pd
import hail as hl

# ----------------------------------------------------------------------------
# 1. Get all individuals with ANY AIP diagnostic code (≥1)
# ----------------------------------------------------------------------------
ht = hl.read_table(f'{my_bucket}/tmp/ICD_codes.ht')
person_df = hl.read_table(f'{my_bucket}/tmp/hmbs_demog.ht')
ht = ht.filter(hl.is_defined(person_df[hl.int32(ht.person_id)]))

# Count AIP codes per person
aip_codes = ht.filter(ht.ICD == 'E80.21')
aip_counts = aip_codes.group_by('person_id').aggregate(
    n_aip_codes = hl.agg.count()
)
aip_df = aip_counts.to_pandas()
aip_df['person_id'] = aip_df['person_id'].astype(int)

# ----------------------------------------------------------------------------
# 2. Get qualifying variant carriers (AC < 6, meets pathogenicity criteria)
# ----------------------------------------------------------------------------
qualifying_carriers = hl.read_table(f'{my_bucket}/tmp/carriers.ht')
qualifying_carrier_ids = set(qualifying_carriers.person_id.collect())

# ----------------------------------------------------------------------------
# 3. Get ALL ClinVar P/LP carriers (including high-frequency)
# ----------------------------------------------------------------------------
mt = hl.read_matrix_table(f'{my_bucket}/tmp/hmbs_annotated.mt')
mt = mt.filter_rows(mt.clinvar_path == 1)

# Annotate with AC for flagging high-frequency
mt = mt.annotate_rows(
    ac_filtered = hl.agg.count_where(mt.GT.is_non_ref() & (mt.FT != 'FAIL'))
)

# Get all carriers of any ClinVar P/LP variant
mt = mt.annotate_cols(
    has_any_clinvar_plp = hl.agg.any(mt.GT.is_non_ref() & (mt.FT != 'FAIL')),
    has_highfreq_clinvar_plp = hl.agg.any(
        mt.GT.is_non_ref() & (mt.FT != 'FAIL') & (mt.ac_filtered > 5)
    )
)

clinvar_carriers = mt.cols()
clinvar_carriers = clinvar_carriers.filter(clinvar_carriers.has_any_clinvar_plp)
clinvar_df = clinvar_carriers.select('has_any_clinvar_plp', 'has_highfreq_clinvar_plp').to_pandas()
clinvar_df = clinvar_df.rename(columns={'s': 'person_id'})
clinvar_df['person_id'] = clinvar_df['person_id'].astype(int)

# ----------------------------------------------------------------------------
# 4. Get PBG testing data
# ----------------------------------------------------------------------------
# Note: Assumes dataset_43814451_measurement_df exists from prior query

labs = (
    dataset_43814451_measurement_df
    .groupby(['person_id', 'standard_concept_name'])
    .agg(max_value=('value_as_number', 'max'))
    .reset_index()
)

reference_ranges = {
    'Porphobilinogen/Creatinine [Mass Ratio] in Urine': 2.0,
    'Porphobilinogen [Moles/volume] in Urine': 8.8,
    'Porphobilinogen [Mass/volume] in Urine': 2.0,
    'Porphobilinogen [Mass/time] in 24 hour Urine': 2.0,
    'Porphobilinogen [Moles/time] in 24 hour Urine': 8.8
}

# Exclude qualitative test
labs = labs[labs['standard_concept_name'] != 'Porphobilinogen [Presence] in Urine']

# Determine if elevated
labs['upper_limit'] = labs['standard_concept_name'].map(reference_ranges)
labs['elevated'] = labs['max_value'] > labs['upper_limit']

# Collapse to one row per person
labs_summary = labs.groupby('person_id').agg(
    any_elevated = ('elevated', 'any')
).reset_index()

labs_summary['pbg_result'] = labs_summary['any_elevated'].map({True: 'Elevated', False: 'WNL'})
tested_ids = set(labs_summary['person_id'].tolist())

# ----------------------------------------------------------------------------
# 5. Merge everything together
# ----------------------------------------------------------------------------

# Start with union of all relevant person_ids
all_ids = set(aip_df['person_id'].tolist())
all_ids.update(qualifying_carrier_ids)
all_ids.update(clinvar_df['person_id'].tolist())
all_ids.update(labs_summary['person_id'].tolist())

master_df = pd.DataFrame({'person_id': list(all_ids)})

# Merge AIP code counts
master_df = master_df.merge(aip_df, on='person_id', how='left')

# Add qualifying carrier flag
master_df['qualifying_variant'] = master_df['person_id'].isin(qualifying_carrier_ids)

# Merge ClinVar carrier info
master_df = master_df.merge(clinvar_df, on='person_id', how='left')
master_df['has_any_clinvar_plp'] = master_df['has_any_clinvar_plp'].fillna(False)
master_df['has_highfreq_clinvar_plp'] = master_df['has_highfreq_clinvar_plp'].fillna(False)

# Merge PBG testing
master_df = master_df.merge(
    labs_summary[['person_id', 'pbg_result']], 
    on='person_id', 
    how='left'
)
master_df['pbg_result'] = master_df['pbg_result'].fillna('Not tested')

# ----------------------------------------------------------------------------
# 6. Add PPOX variant column (placeholder - fill in ID)
# ----------------------------------------------------------------------------
PPOX_CARRIER_ID = None  # <-- Replace with actual person_id availble in workspace version
master_df['ppox_variant'] = master_df['person_id'] == PPOX_CARRIER_ID

# ----------------------------------------------------------------------------
# 7. Filter: Remove ClinVar P/LP carriers without testing or AIP diagnosis
# ----------------------------------------------------------------------------
# Keep individual if:
#   - Has qualifying variant, OR
#   - Has at least 1 AIP code, OR
#   - Has PBG testing, OR
#   - Has PPOX variant, OR
#   - Does NOT have ClinVar P/LP (i.e., they're in table for other reasons)

keep_mask = (
    master_df['qualifying_variant'] |
    (master_df['n_aip_codes'] >= 1) |
    (master_df['pbg_result'] != 'Not tested') |
    master_df['ppox_variant'] |
    (~master_df['has_any_clinvar_plp'])
)

master_df = master_df[keep_mask].copy()

# ----------------------------------------------------------------------------
# 8. Create display columns and de-identify
# ----------------------------------------------------------------------------

# Categorize AIP diagnosis status
master_df['aip_code_count'] = master_df['n_aip_codes'].apply(
    lambda x: '2+' if x >= 2 else ('1' if x == 1 else '0')
)

master_df = master_df[master_df['n_aip_codes'] > 0]

# Create clean variant status column
def variant_status(row):
    if row['qualifying_variant']:
        return 'Qualifying variant'
    elif row['has_highfreq_clinvar_plp']:
        return 'High-frequency ClinVar P/LP'
    elif row['has_any_clinvar_plp']:
        return 'ClinVar P/LP (excluded by other criteria)'
    else:
        return 'No pathogenic HMBS variant'

master_df['variant_status'] = master_df.apply(variant_status, axis=1)

# De-identify: create consistent individual IDs
master_df = master_df.sort_values('person_id')
master_df['individual_id'] = range(1, len(master_df) + 1)

# Select and order final columns
supp_table_2 = master_df[[
    'individual_id',
    'aip_code_count',
    'variant_status',
    'qualifying_variant',
    'has_any_clinvar_plp',
    'has_highfreq_clinvar_plp',
    'ppox_variant',
    'pbg_result'
]].copy()

# Rename for publication
supp_table_2 = supp_table_2.rename(columns={
    'individual_id': 'Individual',
    'aip_code_count': 'AIP Diagnosis Count',
    'variant_status': 'HMBS Variant Status',
    'qualifying_variant': 'Qualifying HMBS Variant',
    'has_any_clinvar_plp': 'Any ClinVar P/LP HMBS',
    'has_highfreq_clinvar_plp': 'High-Frequency ClinVar P/LP HMBS',
    'ppox_variant': 'PPOX Variant',
    'pbg_result': 'PBG Result'
})

supp_table_2.to_csv('SupplementaryTable2.csv', index=False)

# Analysis of variants in CPOX and PPOX

In [ ]:
pid_carriers = hl.read_table(f'{my_bucket}/tmp/carriers.ht')
pid_carriers = set(pid_carriers.person_id.collect())

ht = hl.read_table(f'{my_bucket}/tmp/ICD_codes.ht')
person_df = hl.read_table(f'{my_bucket}/tmp/hmbs_demog.ht')
ht = ht.filter(hl.is_defined(person_df[hl.int32(ht.person_id)]))
aip_cases_all = ht.aggregate(hl.agg.filter(ht.ICD == 'E80.21', hl.agg.collect_as_set(ht.person_id)))
aip_cases = aip_cases_all - pid_carriers

mt = hl.read_matrix_table('gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt')
hmbs = hl.read_table(f'{my_bucket}/tmp/cptA.ht')
clinvar_data = hl.read_table(f'{my_bucket}/tmp/clinvar.ht')

mt = mt.filter_rows(
    (#CPOX
        (mt.locus.contig == 'chr3') & 
            (
                (mt.locus.position >= (98579446)) & 
                (mt.locus.position <= (98594101))
            )
    ) | (#PPOX
        (mt.locus.contig == 'chr1') & 
            (
                (mt.locus.position >= (161166301)) & 
                (mt.locus.position <= (161171218))
            )
    ) | (#HMBS
        (mt.locus.contig == 'chr11') & 
            (
                (mt.locus.position >= (119084294)) & 
                (mt.locus.position <= (119093549))
            )
    )   
)

mt = mt.checkpoint(f'{my_bucket}/tmp/other.ht', overwrite=True)


mt = mt.annotate_rows(
    cnt = hl.agg.count_where(mt.GT.is_non_ref() & (mt.FT != 'FAIL'))
)


mt = mt.filter_cols(
    hl.set(aip_cases).contains(hl.int(mt.s))
)

mt = mt.annotate_rows(
    ids = hl.agg.filter(mt.GT.is_non_ref() & (mt.FT != 'FAIL'), hl.agg.collect_as_set(hl.int(mt.s)))
)

mt = mt.filter_rows(
    hl.agg.any(mt.GT.is_non_ref() & (mt.FT != 'FAIL'))
)

mt.row.show(40)

# Filtering of SCHEMA/BiPEX Data

In [ ]:
sc = hl.spark_context()
spark = SparkSession(sc)
collapsing_vars = (spark
 .read
 .parquet(f'{my_bucket}/tmp/HMBS_Combined.parquet')
)

collapsing_vars = hl.Table.from_spark(collapsing_vars)

collapsing_vars = collapsing_vars.key_by(
    locus = hl.locus(
        'chr' + collapsing_vars['locus.contig'].split('-')[0],
        hl.int32(collapsing_vars['locus.contig'].split('-')[1]),
        reference_genome='GRCh38'
    ),
    alleles = collapsing_vars.alleles
)


aou = hl.read_table(f'{my_bucket}/tmp/cpt.ht')
collapsing_vars = collapsing_vars.annotate(
    aou_ac = aou[collapsing_vars.key].gvs_all_ac
)

collapsing_vars = collapsing_vars.filter(
    (hl.int(collapsing_vars.aou_ac) < 6) | 
    hl.is_missing(collapsing_vars.aou_ac) 
).drop('aou_ac')

collapsing_vars = collapsing_vars.to_spark()

(collapsing_vars
 .repartition(1)
 .write
 .mode('overwrite')
 .parquet(f'{my_bucket}/tmp/HMBS_Combined_AoUFilt.parquet')
)

!gsutil -m cp -r gs://fc-secure-57a37491-77aa-498c-b0f0-2163b5feb12e/tmp/HMBS_Combined_AoUFilt.parquet .